# 03_6b. Test: sensitivity checks and score stability / Test: проверки чувствительности и стабильность скора

**EN.** Second part of the single test opening, under the same frozen protocol: it reads the test predictions saved by `03_6` (`data/processed/predictions/test_frame_y1.parquet`) in a fresh process, so the run fits into 8 GB of memory. No model is refitted or recalibrated here.

**RU.** Вторая часть единственного открытия test по тому же замороженному протоколу: ноутбук читает прогнозы test, сохранённые в `03_6` (`data/processed/predictions/test_frame_y1.parquet`), в отдельном процессе, чтобы расчёт поместился в 8 ГБ памяти. Модели здесь не переобучаются и не перекалибровываются.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import gc
import numpy as np
import pandas as pd
import polars as pl
pd.set_option("display.width", 230)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

from xai_ews.config import data_config, feature_config
from xai_ews.evaluation.compare import h4_check
from xai_ews.evaluation.el import assign_lgd
from xai_ews.evaluation.metrics import auc, ks
from xai_ews.evaluation.protocol import check_frozen
from xai_ews.evaluation.stats import delong_test, one_snapshot_per_loan, score_psi
from xai_ews.models import registry
from xai_ews.models.data import load_eval, load_train
from xai_ews.paths import MODELS_DIR, PREDICTIONS_DIR, MODELING_DIR

SPLIT = "test"               # "validation" only for the dry run of the code before the protocol is frozen
OUT = MODELING_DIR / ("test" if SPLIT == "test" else "dry_run_validation")
if SPLIT == "test":
    print(check_frozen())    # stops here if the protocol is not frozen or a hash differs
SEED, B = 20261025, 1000
DCFG, SEL = data_config(), feature_config()["selected"]
def mem():
    import resource
    return f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9:.1f} GB peak"

TASK = "y1"
MAIN = ["b0_y1", "m1_y1", "m2_y1"]
GBDT = ["xgb_y1", "lgb_y1"]
SENS_THRESHOLDS = DCFG["alert"]["pd_threshold_sensitivity"]
frame = pl.read_parquet(PREDICTIONS_DIR / f"{SPLIT}_frame_{TASK}.parquet")
leadtime = pd.read_csv(OUT / "lead_time.csv")
print(frame.shape, mem())

protocol frozen, 79 files verified: **Статус: ЗАМОРОЖЕН 2026-11-03 10:54** — второе открытие test (раздел 0). Протокол пишется до открытия test; пробный прогон кода 03_6 / 03_6b сделан на validation 2026-11-03 (все ячейки выполнены, результаты — `reports/modeling/dry_run_validation/`; в нём калибраторы оцениваются и на половине B, поэтому цифры — только проверка кода). Test открывается один раз ноутбуками `notebooks/03_6_test_evaluation.ipynb` и `notebooks/03_6b_test_sensitivity.ipynb`; их первая ячейка сверяет sha256 из раздела 12. Любое изменение после просмотра test помечается как post-hoc и записывается в раздел 13 и в журнал решений.


(4111592, 35) 1.0 GB peak


### 1. Sensitivity of the models / Чувствительность моделей

**EN.** Every check reuses the main hyperparameters (an assumption named in the report): r = 0.1; linear instead of logarithmic ω; without the rows of loans prepaid within the window (train and test); only the 20 % holdout loans (loan-disjoint from train); only the new cohorts 2015–2018; three variants of the disaster rule. AUC and KS on raw logits.

**RU.** Все проверки используют гиперпараметры основных моделей (допущение, называется в отчёте): r = 0,1; линейный вместо логарифмического ω; без строк кредитов, досрочно погашенных в окне (в train и test); только отложенные 20 % кредитов (не пересекаются с train); только новые когорты 2015–2018; три варианта правила бедствий. AUC и KS — по сырым логитам.

In [2]:
rows = []
def add(check, d, models):
    y = d[TASK].to_numpy()
    if len(np.unique(y)) < 2:              # e.g. no holdout loans in validation (dry run)
        return
    for n in models:
        z = d[f"z_{n}"].to_numpy()
        rows.append({"check": check, "model": n, "rows": d.height, "events": int(y.sum()), "auc": auc(y, z), "ks": ks(y, z)})
main = frame.filter(~pl.col("excl_main"))
add("main", main, [*MAIN, *GBDT])
add("r = 0.1", main, ["m1_y1_r01"])
add("linear omega", main, ["m1_y1_linear"])
add("without prepaid rows (train and test)", main.filter(~pl.col("prepaid_in_window").fill_null(False)),
    ["m1_y1", "m1_y1_noprepaid"])
add("holdout 20 % loans (loan-disjoint)", main.filter(pl.col("holdout_loan")), [*MAIN, *GBDT])
add("new cohorts 2015-2018", main.filter(pl.col("new_cohort")), [*MAIN, *GBDT])
add("disaster rule: all rows", frame, [*MAIN, *GBDT])
add("disaster rule: hurricanes only", frame.filter(~pl.col("excl_hurricane")), [*MAIN, *GBDT])
add("disaster rule: any ZIP3 overlap", frame.filter(~pl.col("excl_any_overlap")), [*MAIN, *GBDT])
sens_models = pd.DataFrame(rows)
sens_models.to_csv(OUT / "sensitivity_models.csv", index=False)
sens_models.set_index(["check", "model"])

rows  events    auc     ks
check                                 model                                         
main                                  b0_y1            3758886    7824 0.8247 0.5029
                                      m1_y1            3758886    7824 0.8378 0.5268
                                      m2_y1            3758886    7824 0.8335 0.5211
                                      xgb_y1           3758886    7824 0.8390 0.5324
                                      lgb_y1           3758886    7824 0.8387 0.5297
r = 0.1                               m1_y1_r01        3758886    7824 0.8366 0.5260
linear omega                          m1_y1_linear     3758886    7824 0.8371 0.5247
without prepaid rows (train and test) m1_y1            3383787    7156 0.8455 0.5414
                                      m1_y1_noprepaid  3383787    7156 0.8454 0.5444
holdout 20 % loans (loan-disjoint)    b0_y1             748575    1594 0.8436 0.5320
                                      m1_y1             748575    1594 0.8497 0.5446
                                      m2_y1             748575    1594 0.8478 0.5461
                                      xgb_y1            748575    1594 0.8497 0.5501
                                      lgb_y1            748575    1594 0.8511 0.5515
new cohorts 2015-2018                 b0_y1            2962092    6482 0.8259 0.5047
                                      m1_y1            2962092    6482 0.8395 0.5303
                                      m2_y1            2962092    6482 0.8347 0.5256
                                      xgb_y1           2962092    6482 0.8396 0.5351
                                      lgb_y1           2962092    6482 0.8393 0.5314
disaster rule: all rows               b0_y1            4111592    8661 0.8219 0.4986
                                      m1_y1            4111592    8661 0.8347 0.5222
                                      m2_y1            4111592    8661 0.8307 0.5160
                                      xgb_y1           4111592    8661 0.8356 0.5252
                                      lgb_y1           4111592    8661 0.8355 0.5229
disaster rule: hurricanes only        b0_y1            4077470    8481 0.8232 0.5005
                                      m1_y1            4077470    8481 0.8363 0.5248
                                      m2_y1            4077470    8481 0.8323 0.5182
                                      xgb_y1           4077470    8481 0.8373 0.5286
                                      lgb_y1           4077470    8481 0.8371 0.5259
disaster rule: any ZIP3 overlap       b0_y1            3482848    7204 0.8241 0.5011
                                      m1_y1            3482848    7204 0.8373 0.5261
                                      m2_y1            3482848    7204 0.8331 0.5200
                                      xgb_y1           3482848    7204 0.8388 0.5321
                                      lgb_y1           3482848    7204 0.8386 0.5297

**EN. How to read.** The order B0 < M2 < M1 ≈ XGBoost ≈ LightGBM holds in every check. r = 0.1: M1 0.837 (−0.001); linear ω: 0.837 (−0.0006). Without the rows of loans prepaid in the window AUC rises to 0.846 for M1, and the model trained without them gives the same (0.845): prepaid rows are harder to rank, removing them from training changes nothing. Holdout loans (never seen in train; 748,575 rows, 1,594 events): M1 0.850, B0 0.844, XGBoost 0.850, LightGBM 0.851 — no sign that the models remember train loans. New cohorts 2015–2018: M1 0.839, as on the whole test. The three disaster-rule variants: M1 0.835–0.837.

**RU. Как читать.** Порядок B0 < M2 < M1 ≈ XGBoost ≈ LightGBM сохраняется во всех проверках. r = 0,1: M1 0,837 (−0,001); линейный ω: 0,837 (−0,0006). Без строк кредитов, досрочно погашенных в окне, AUC M1 растёт до 0,846, и модель, обученная без них, даёт то же (0,845): досрочно погашаемые кредиты труднее ранжировать, но их удаление из обучения ничего не меняет. Отложенные кредиты (не встречались в train; 748 575 строк, 1 594 события): M1 0,850, B0 0,844, XGBoost 0,850, LightGBM 0,851 — признаков того, что модели «запомнили» кредиты train, нет. Новые когорты 2015–2018: M1 0,839, как на всём test. Три варианта правила бедствий: M1 0,835–0,837.

### 2. Sensitivity of H4 / Чувствительность H4

**EN.** The H4 rule (protocol) repeated with: linear ω; alert thresholds 0.02 and 0.005; isotonic instead of Platt calibration; four LGD variants (after mortgage insurance, undiscounted, train vintages only, constant); three disaster-rule variants.

**RU.** Правило H4 (протокол) повторяется: с линейным ω; с порогами тревоги 0,02 и 0,005; с изотонической калибровкой вместо Платта; с четырьмя вариантами LGD (после страхования, без дисконтирования, только поколения train, постоянный); с тремя вариантами правила бедствий.

In [3]:
h4_rows = {}
# only the columns H4 needs: copies of the full 4 M-row frame per variant do not fit in 8 GB
slim = frame.select("loan_identifier", "m", TASK, "upb", "lgd", "excl_main", "excl_hurricane", "excl_any_overlap",
                    "pd_m1_y1", "pd_m1_y1_noomega", "pd_m1_y1_linear", "z_m1_y1", "z_m1_y1_noomega")
d1 = slim.filter(~pl.col("excl_main"))
h4_rows["main"] = h4_check(d1, TASK, "pd_m1_y1", "pd_m1_y1_noomega", b=B, seed=SEED)
h4_rows["linear omega"] = h4_check(d1, TASK, "pd_m1_y1_linear", "pd_m1_y1_noomega", b=B, seed=SEED)
for thr in SENS_THRESHOLDS:
    h4_rows[f"PD >= {thr}"] = h4_check(d1, TASK, "pd_m1_y1", "pd_m1_y1_noomega", threshold=thr, b=B, seed=SEED)
iso = {n: registry.calibrator(n, "isotonic").predict(d1[f"z_{n}"].to_numpy()) for n in ("m1_y1", "m1_y1_noomega")}
d_iso = d1.with_columns(pl.Series("iso_a", iso["m1_y1"]), pl.Series("iso_b", iso["m1_y1_noomega"]))
h4_rows["isotonic calibration"] = h4_check(d_iso, TASK, "iso_a", "iso_b", b=B, seed=SEED)
lgd_src = load_eval(SPLIT, TASK, [], [], exclusion="main").select("loan_identifier", "m", "mtm_ltv", "property_state")
for v in ("after_mi", "undiscounted", "train_vintages", "constant"):
    lg = lgd_src.with_columns(assign_lgd(lgd_src, DCFG, v).alias(f"lgd_{v}")).select("loan_identifier", "m", f"lgd_{v}")
    h4_rows[f"LGD {v}"] = h4_check(d1.join(lg, on=["loan_identifier", "m"], how="left", maintain_order="left"), TASK,
                                   "pd_m1_y1", "pd_m1_y1_noomega", lgd_col=f"lgd_{v}", b=B, seed=SEED)
    del lg; gc.collect()
for rule, col in (("disaster rule: all rows", None), ("disaster rule: hurricanes only", "excl_hurricane"),
                  ("disaster rule: any ZIP3 overlap", "excl_any_overlap")):
    f = slim if col is None else slim.filter(~pl.col(col))
    h4_rows[rule] = h4_check(f, TASK, "pd_m1_y1", "pd_m1_y1_noomega", b=B, seed=SEED)
    del f; gc.collect()
del slim, d1, d_iso, lgd_src; gc.collect()
sens_h4 = pd.DataFrame(h4_rows).T
sens_h4.to_csv(OUT / "sensitivity_h4.csv")
sens_h4[["el_share_with", "el_share_without", "delta", "delta_lo", "delta_hi", "d_slope", "d_intercept", "verdict"]]

,el_share_with,el_share_without,delta,delta_lo,delta_hi,d_slope,d_intercept,verdict
main,0.2717,0.2766,-0.0049,-0.0087,-0.0011,0.0017,0.0011,partial
linear omega,0.2693,0.2766,-0.0073,-0.0121,-0.0022,0.0025,0.0005,partial
PD >= 0.02,0.1090,0.1097,-0.0007,-0.0040,0.0022,0.0017,0.0011,partial
PD >= 0.005,0.5114,0.5156,-0.0042,-0.0090,0.0004,0.0017,0.0011,partial
isotonic calibration,0.3195,0.3028,0.0167,0.0124,0.0212,0.0040,0.0000,confirmed
LGD after_mi,0.2674,0.2725,-0.0050,-0.0087,-0.0013,0.0017,0.0011,partial
LGD undiscounted,0.2723,0.2772,-0.0049,-0.0087,-0.0011,0.0017,0.0011,partial
LGD train_vintages,0.2673,0.2722,-0.0049,-0.0085,-0.0012,0.0017,0.0011,partial
LGD constant,0.2651,0.2699,-0.0048,-0.0083,-0.0012,0.0017,0.0011,partial
disaster rule: all rows,0.2668,0.2708,-0.0041,-0.0075,-0.0008,0.0019,0.0012,partial


**EN. How to read.** The calibration part holds in all 12 variants (|Δ slope|, |Δ intercept| ≤ 0.004). The EL part fails in 11 of 12: Δ is negative with the CI below 0 for linear ω, all four LGD variants and the three disaster rules (−0.4 … −0.7 pp), and the CI includes 0 at the thresholds 0.02 and 0.005. Only with isotonic calibration is Δ positive, +1.7 pp [1.2; 2.1], and H4 confirmed. Isotonic calibration is a step function, so the list at 0.01 is made of whole steps and its size differs between the two models; the result depends on where the steps fall. Conclusion: the exposure weight does not raise the captured EL robustly; the main (Platt) result — H4 partial — stands.

**RU. Как читать.** Часть про калибровку выполнена во всех 12 вариантах (|Δ slope|, |Δ intercept| ≤ 0,004). Часть про EL не выполнена в 11 из 12: Δ отрицательна, и интервал ниже 0 при линейном ω, всех четырёх вариантах LGD и трёх правилах бедствий (−0,4 … −0,7 п.п.), а при порогах 0,02 и 0,005 интервал включает 0. Только с изотонической калибровкой Δ положительна, +1,7 п.п. [1,2; 2,1], и H4 подтверждается. Изотоническая калибровка — ступенчатая функция, поэтому список при 0,01 состоит из целых ступеней, и его размер у двух моделей разный; результат зависит от того, где проходят ступени. Вывод: вес по ОД не повышает пойманный EL устойчиво; основной результат (с Платтом) — H4 подтверждена частично — остаётся.

### 3. Sensitivity of H3 / Чувствительность H3

**EN.** ΔAUC of the full vs the static H3 model under the three disaster-rule variants, DeLong on one snapshot per loan and on all snapshots; the median lead time of the full model at the alert thresholds 0.02 and 0.005 (from `03_6`).

**RU.** ΔAUC полной и статической моделей H3 при трёх вариантах правила бедствий, DeLong на одном снимке на кредит и на всех снимках; медианный lead time полной модели при порогах тревоги 0,02 и 0,005 (из `03_6`).

In [4]:
h3_rows = {}
for rule, col in (("main", "excl_main"), ("disaster rule: all rows", None), ("disaster rule: hurricanes only", "excl_hurricane"),
                  ("disaster rule: any ZIP3 overlap", "excl_any_overlap")):
    f = frame if col is None else frame.filter(~pl.col(col))
    y, zf, zs = f[TASK].to_numpy(), f["z_h3_full_y1"].to_numpy(), f["z_h3_static_y1"].to_numpy()
    one = one_snapshot_per_loan(f["loan_identifier"].to_numpy(), seed=SEED)
    h3_rows[rule] = {"delta_auc": auc(y, zf) - auc(y, zs), "delong_p_one_snapshot": delong_test(y[one], zf[one], zs[one])["p"],
                     "delong_p_all_snapshots": delong_test(y, zf, zs)["p"]}
for thr in SENS_THRESHOLDS:
    h3_rows[f"lead time, PD >= {thr}"] = leadtime[(leadtime["model"] == "h3_full_y1") & (leadtime["threshold"] == thr)][
        ["median_lead", "median_lo", "median_hi", "alert_rate"]].iloc[0].to_dict()
sens_h3 = pd.DataFrame(h3_rows).T
sens_h3.to_csv(OUT / "sensitivity_h3.csv")
sens_h3

,delta_auc,delong_p_one_snapshot,delong_p_all_snapshots,median_lead,median_lo,median_hi,alert_rate
main,0.0152,0.0000,0.0000,NaN,NaN,NaN,NaN
disaster rule: all rows,0.0151,0.0000,0.0000,NaN,NaN,NaN,NaN
disaster rule: hurricanes only,0.0151,0.0000,0.0000,NaN,NaN,NaN,NaN
disaster rule: any ZIP3 overlap,0.0160,0.0000,0.0000,NaN,NaN,NaN,NaN
"lead time, PD >= 0.02",NaN,NaN,NaN,10.0000,9.0000,10.0000,0.1623
"lead time, PD >= 0.005",NaN,NaN,NaN,10.0000,10.0000,10.0000,0.5987


**EN. How to read.** H3 does not depend on the disaster rule: ΔAUC full − static is +0.0151 … +0.0160 in all three variants, DeLong p < 10⁻⁴⁰ on one snapshot per loan and < 10⁻⁷⁰ on all snapshots. The median lead time of the full model is 10 months at PD ≥ 0.02 (CI [9; 10]) and at 0.005 ([10; 10]), above 3 in both.

**RU. Как читать.** H3 не зависит от правила бедствий: ΔAUC «полная − статическая» +0,0151 … +0,0160 во всех трёх вариантах, p DeLong < 10⁻⁴⁰ на одном снимке на кредит и < 10⁻⁷⁰ на всех снимках. Медианный lead time полной модели — 10 месяцев при PD ≥ 0,02 (интервал [9; 10]) и при 0,005 ([10; 10]), в обоих случаях выше 3.

### 4. Score stability train → test / Стабильность скора train → test

**EN.** **PSI** (population stability index) of each model's raw score between train (weighted back to the full population by 1/r) and test: < 0.1 stable, 0.1–0.25 moderate shift, > 0.25 large shift. For the boosted models the train scores are their out-of-fold logits.

**RU.** **PSI** (индекс стабильности популяции) сырого скора каждой модели между train (с весом 1/r, то есть приведённым к полной популяции) и test: < 0,1 — стабильно, 0,1–0,25 — умеренный сдвиг, > 0,25 — сильный сдвиг. Для бустинговых моделей скор train — их out-of-fold логиты.

In [5]:
test_scores = frame.filter(~pl.col("excl_main")).select([c for c in frame.columns if c.startswith("z_")])
del frame, main; gc.collect()
psi = []
for n in ("m1_y1", *GBDT):                 # out-of-fold train logits saved with the fold models
    oof = pl.read_parquet(MODELS_DIR / n / "oof.parquet")
    psi.append({"model": n, "psi": score_psi(oof["logit"].to_numpy(), test_scores[f"z_{n}"].to_numpy(),
                                             oof["sample_weight"].to_numpy())})
tr = load_train(TASK, SEL["features"], SEL["categorical"])
for n in ("b0_y1", "m2_y1"):
    psi.append({"model": n, "psi": score_psi(registry.predict_any(n, tr), test_scores[f"z_{n}"].to_numpy(),
                                             tr["sample_weight"].to_numpy())})
del tr; gc.collect()
psi = pd.DataFrame(psi).set_index("model")
psi.to_csv(OUT / "score_psi.csv")
psi

,psi
model,
m1_y1,0.0895
xgb_y1,0.0886
lgb_y1,0.0917
b0_y1,0.0609
m2_y1,0.0982


**EN. Summary.** The test results are robust. The model order holds for r = 0.1, linear ω, without prepaid rows, on loans never seen in train, on the new cohorts and under every disaster rule. H3 holds under every disaster rule and at every alert threshold. H4 stays partial: calibration is preserved everywhere, the captured EL rises only with isotonic calibration. PSI train → test is 0.06 (B0) … 0.10 (M2), CatBoost / XGBoost / LightGBM 0.09: the score distribution is stable (M2 at the border of a moderate shift).

**RU. Итог.** Результаты test устойчивы. Порядок моделей сохраняется при r = 0,1, линейном ω, без досрочно погашенных строк, на кредитах, не встречавшихся в train, на новых когортах и при любом правиле бедствий. H3 держится при любом правиле бедствий и любом пороге тревоги. H4 остаётся частичной: калибровка сохраняется везде, пойманный EL растёт только при изотонической калибровке. PSI train → test — от 0,06 (B0) до 0,10 (M2), CatBoost / XGBoost / LightGBM — 0,09: распределение скора стабильно (M2 — на границе умеренного сдвига).